In [1]:
pip install optlang

Note: you may need to restart the kernel to use updated packages.


In [2]:
import jupyter_utils as ju
model = ju.load_model("Experiment", "250521_iABA974_kegg_bjp.sbml")

In [3]:
candidate_reactions = []
for rxn in model.reactions:
    if "nad_c" in [met.id for met in rxn.metabolites] or "nadp_c" in [met.id for met in rxn.metabolites]:
        candidate_reactions.append(rxn.id)

In [5]:
from copy import deepcopy

for rxn_id in candidate_reactions:
    original_rxn = model.reactions.get_by_id(rxn_id)
    swapped_rxn = deepcopy(original_rxn)
    swapped_rxn.id = original_rxn.id + "_NADP"

    # Replace NAD with NADP and NADH with NADPH
    for met in list(swapped_rxn.metabolites):
        if met.id == "nad_c":
            swapped_rxn.add_metabolites({model.metabolites.nad_c: -1, model.metabolites.nadp_c: 1})
        elif met.id == "nadh_c":
            swapped_rxn.add_metabolites({model.metabolites.nadh_c: -1, model.metabolites.nadph_c: 1})

    model.add_reactions([swapped_rxn])

In [9]:
M = 1000
interface = model.solver.interface  # e.g., glpk_interface

for rxn_id in candidate_reactions:
    rxn1 = model.reactions.get_by_id(rxn_id)
    rxn2 = model.reactions.get_by_id(rxn_id + "_NADP")

    # Create binary switch variable using the model's actual solver
    y = interface.Variable(name="y_" + rxn_id, type="binary")

    # Add the variable and constraints to the solver, not the model directly
    model.solver._add_variable(y)

    # Add constraints: v1 ≤ M(1 - y), v2 ≤ My
    model.solver._add_constraint(
        interface.Constraint(rxn1.forward_variable - M * (1 - y), ub=0, name=rxn1.id + "_fwd_switch")
    )
    model.solver._add_constraint(
        interface.Constraint(rxn1.reverse_variable - M * (1 - y), ub=0, name=rxn1.id + "_rev_switch")
    )
    model.solver._add_constraint(
        interface.Constraint(rxn2.forward_variable - M * y, ub=0, name=rxn2.id + "_fwd_switch")
    )
    model.solver._add_constraint(
        interface.Constraint(rxn2.reverse_variable - M * y, ub=0, name=rxn2.id + "_rev_switch")
    )

In [20]:
# set reactions bounds for a specific reaction 
reaction = model.reactions.get_by_id('EX_h2_e')
reaction.bounds = -100.0, 0.0

reaction = model.reactions.get_by_id('EX_o2_e')
reaction.bounds = -100, 0.0

reaction = model.reactions.get_by_id('EX_co2_e')
reaction.bounds = -100.0, 0.0

In [12]:
for var in model.solver.variables:
    if var.name.startswith("y_"):
        print(var.name, var.primal)

y_13PPDH 0.0
y_1P2CBXLR 0.0
y_2DGLCNRx 0.0
y_2DGLCNRy 0.0
y_2DGULRGx 0.0
y_2DGULRGy 0.0
y_2DGULRx 0.0
y_2DGULRy 0.0
y_3HOXTPP 0.0
y_3OAR40 0.0
y_4ABUTD 0.0
y_4HBADH 0.0
y_4HOXPACMON 0.0
y_4OXPTCOADHx 0.0
y_4OXPTCOADHy 0.0
y_5DGLCNR 0.0
y_5DKGR 0.0
y_6HNACMO 0.0
y_AAAN 0.0
y_AACOAR_syn 0.0
y_AALDH 0.0
y_AASAD3 0.0
y_ABUTD 0.0
y_ACOAD1 0.0
y_ACOAD2 0.0
y_ACOAD3 0.0
y_ACOAD4_1 0.0
y_ACOAD5_1 0.0
y_ACOAD6 0.0
y_ACOAD7 0.0
y_ACTD 0.0
y_ACTD2 0.0
y_ACTD_1 0.0
y_ACTDa 0.0
y_AGPR 0.0
y_AHGDx 0.0
y_AKGDH 0.0
y_ALCD19 0.0
y_ALCD19y 0.0
y_ALCD2ir 0.0
y_ALCD2x 0.0
y_ALCD2y 0.0
y_ALCD4 0.0
y_ALDD1 0.0
y_ALDD19xr 0.0
y_ALDD20x 0.0
y_ALDD20y 0.0
y_ALDD2x 0.0
y_ALDD2y 0.0
y_ALDD3 0.0
y_ALDD31 0.0
y_ALDD31_1 0.0
y_ALDD3y 0.0
y_ALDD4 0.0
y_ALDD4x 0.0
y_ALDD5 0.0
y_ALDD6 0.0
y_ALDH_1 0.0
y_ALR2 0.0
y_ALR3 0.0
y_ALR4x 0.0
y_AMPMS2 0.0
y_ANDO1 0.0
y_APPLDHr 0.0
y_APRAUR 0.0
y_ARABR 0.0
y_ARABRr 0.0
y_ASAD 0.0
y_ATHRDHr 0.0
y_BDH 0.0
y_BETALDHx 0.0
y_BETALDHy 0.0
y_BNOCA 0.0
y_BZDH 0.0
y_CHOLD 0.0
y_CMCMSAD

In [21]:
# Constrain biomass to at least 10% of WT
biomass_rxn = model.reactions.get_by_id("Growth")
model.objective = biomass_rxn
solution = model.optimize()
print("Max product flux:", solution.objective_value)
for var in model.solver.variables:
    if var.name.startswith("y_") and round(var.primal, 2) == 1:
        print("Switch cofactor for:", var.name)

Max product flux: 0.45181389948896145
Switch cofactor for: y_ALCD19
Switch cofactor for: y_GCALDD


In [18]:
from optlang.symbolics import Zero
interface = model.solver.interface

binary_vars = [v for v in model.solver.variables if v.name.startswith("y_")]
sum_y = sum(binary_vars)
model.solver._add_constraint(interface.Constraint(sum_y, ub=3, name="max_switches"))